# Candidate Pool Merge

Ноутбук объединяет base retrieval candidates с DL-кандидатами DSSM, BERT4Rec и SASRec.

Пайплайн:
- загрузка candidate sources;
- validation-сравнение разных quota-конфигураций;
- расчет recall/NDCG и unique positive contribution;
- материализация выбранного candidate pool для реранкера.

Обучение DL-моделей здесь не запускается.

In [1]:
from __future__ import annotations

import gc
import json
import shutil
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import trange, tqdm

## Config

In [ ]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "datamart").exists() or (p / "data" / "raw").exists() or (p / "data").exists():
            return p
    raise FileNotFoundError("Не найден project root с data/datamart или data/raw")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATAMART_DIR = DATA_DIR / "datamart"
AGG_DIR = DATAMART_DIR / "agg"
CANDIDATES_DIR = DATAMART_DIR / "candidates"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class FourSourceMergeConfig:
    # Validation target window.
    val_target_start: date = date(2025, 6, 18)
    val_target_end: date = date(2025, 7, 1)

    n_shards: int = 128

    base_source_name: str = "base_non_dl"
    base_dir_name: str = "merged"
    dssm_source_name: str = "dl_dssm_final"
    bert_source_name: str = "dl_bert4rec_small"

# SASRec candidates can be included as an additional DL source.
    use_sasrec: bool = True
    sasrec_source_name: str = "dl_sasrec_v3"

    # Keep policy for experiments.
    base_keep_per_user: int = 700
    dssm_max_keep_per_user: int = 300
    bert_max_keep_per_user: int = 300
    sasrec_max_keep_per_user: int = 300

    # If total cap is None for an experiment, it is set to base + dssm + bert + sasrec quotas.
    metric_ks: tuple[int, ...] = (20, 50, 100, 200, 300, 700, 900, 1000, 1100, 1200, 1300, 1400)

    # Validation experiments. Keep this list small: each row requires one streaming pass over val shards.
    # name, base_k, dssm_k, bert_k, sasrec_k, total_cap
    experiment_grid: tuple[tuple[str, int, int, int, int, int | None], ...] = (
        ("base_700", 700, 0, 0, 0, 700),

        # Single DL-source baselines.
        ("base700_dssm300", 700, 300, 0, 0, 1000),
        ("base700_bert300", 700, 0, 300, 0, 1000),
        ("base700_sasrec300", 700, 0, 0, 300, 1000),

        # Best source + one extra source.
        ("base700_dssm300_bert50", 700, 300, 50, 0, 1050),
        ("base700_dssm300_bert100", 700, 300, 100, 0, 1100),
        ("base700_dssm300_bert200", 700, 300, 200, 0, 1200),
        ("base700_dssm300_bert300", 700, 300, 300, 0, 1300),
        ("base700_dssm300_sasrec50", 700, 300, 0, 50, 1050),
        ("base700_dssm300_sasrec100", 700, 300, 0, 100, 1100),
        ("base700_dssm300_sasrec200", 700, 300, 0, 200, 1200),
        ("base700_dssm300_sasrec300", 700, 300, 0, 300, 1300),

        # Three DL sources together.
        ("base700_dssm300_bert100_sasrec100", 700, 300, 100, 100, 1200),
        ("base700_dssm300_bert200_sasrec100", 700, 300, 200, 100, 1300),
        ("base700_dssm300_bert100_sasrec200", 700, 300, 100, 200, 1300),
        ("base700_dssm300_bert200_sasrec200", 700, 300, 200, 200, 1400),

        # Lower DSSM quota variants in case the final pool is too large.
        ("base700_dssm200_bert200_sasrec100", 700, 200, 200, 100, 1200),
        ("base700_dssm200_bert100_sasrec100", 700, 200, 100, 100, 1100),
    )

    # Materialization for reranker. Disabled by default: first inspect val experiments.
    materialize_selected_config: bool = True
    materialized_output_name: str = "merged_with_dl_all"
    materialize_splits: tuple[str, ...] = ("ranker_train", "val", "test")
    selected_base_k = 700
    selected_dssm_k = 300
    selected_bert_k = 200
    selected_sasrec_k = 100
    selected_total_cap = 1300

    # Reports.
    report_prefix: str = "dl_four_source_merge"


CFG = FourSourceMergeConfig()

print("PROJECT_ROOT:", PROJECT_ROOT)
print(json.dumps({k: str(v) if isinstance(v, date) else v for k, v in asdict(CFG).items()}, ensure_ascii=False, indent=2))

PROJECT_ROOT: c:\Users\denis\Documents\Repo\ozon
{
  "val_target_start": "2025-06-18",
  "val_target_end": "2025-07-01",
  "n_shards": 128,
  "base_source_name": "base_non_dl",
  "base_dir_name": "merged",
  "dssm_source_name": "dl_dssm_final",
  "bert_source_name": "dl_bert4rec_small",
  "use_sasrec": true,
  "sasrec_source_name": "dl_sasrec_v3",
  "base_keep_per_user": 700,
  "dssm_max_keep_per_user": 300,
  "bert_max_keep_per_user": 300,
  "sasrec_max_keep_per_user": 300,
  "metric_ks": [
    20,
    50,
    100,
    200,
    300,
    700,
    900,
    1000,
    1100,
    1200,
    1300,
    1400
  ],
  "experiment_grid": [
    [
      "base_700",
      700,
      0,
      0,
      0,
      700
    ],
    [
      "base700_dssm300",
      700,
      300,
      0,
      0,
      1000
    ],
    [
      "base700_bert300",
      700,
      0,
      300,
      0,
      1000
    ],
    [
      "base700_sasrec300",
      700,
      0,
      0,
      300,
      1000
    ],
    [
      "base

## IO helpers

In [6]:
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def rel(path: Path | str) -> str:
    path = Path(path)
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return str(path)


def path_has_parquet(path: Path) -> bool:
    return path.exists() and any(path.rglob("*.parquet"))


def candidate_files(path: Path) -> list[Path]:
    return sorted(path.rglob("*.parquet")) if path.exists() else []


def candidate_dir_looks_complete(path: Path, *, min_shards: int | None = None) -> bool:
    files = candidate_files(path)
    if not files:
        return False
    if min_shards is None:
        return True
    shard_dirs = {p.parent.name for p in files if p.parent.name.startswith("user_shard=") and p.stat().st_size > 0}
    return len(shard_dirs) >= min_shards


def scan_candidate_shard_or_none(base_dir: Path, shard_id: int) -> pl.LazyFrame | None:
    shard_dir = base_dir / f"user_shard={shard_id}"
    if path_has_parquet(shard_dir):
        return pl.scan_parquet(str(shard_dir / "*.parquet"), hive_partitioning=True)
    # Fallback for non-sharded directories: scan all and filter by hash shard. More expensive but robust.
    if path_has_parquet(base_dir):
        lf = pl.scan_parquet(str(base_dir / "**" / "*.parquet"), hive_partitioning=True)
        return lf.filter((pl.col("user_id").cast(pl.UInt64) % int(CFG.n_shards)) == int(shard_id))
    return None


def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)


def lazy_columns(lf: pl.LazyFrame) -> list[str]:
    try:
        return list(lf.collect_schema().names())
    except Exception:
        return list(lf.schema.keys())


def resolve_manifest_candidate_dir(split: str) -> Path | None:
    manifest_path = CANDIDATES_DIR / "manifest_candidate_generators_enhanced.json"
    if not manifest_path.exists():
        return None
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception:
        return None
    key_by_split = {
        "ranker_train": "ranker_train_candidates",
        "val": "val_candidates",
        "test": "test_candidates",
    }
    raw = manifest.get("outputs", {}).get(key_by_split.get(split, ""))
    if not raw:
        return None
    p = Path(raw)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    return p if candidate_dir_looks_complete(p) else None


def resolve_base_candidate_dir(split: str) -> Path | None:
    p = resolve_manifest_candidate_dir(split)
    if p is not None:
        return p
    for name in [CFG.base_dir_name, "merged", "merged_enhanced", "enhanced_merged"]:
        p = CANDIDATES_DIR / name / split
        if candidate_dir_looks_complete(p):
            return p
    return None


def resolve_source_dir(source_name: str, split: str) -> Path | None:
    p = CANDIDATES_DIR / source_name / split
    return p if candidate_dir_looks_complete(p) else None


def source_state(split: str = "val") -> pd.DataFrame:
    rows = []
    source_map = {
        CFG.base_source_name: resolve_base_candidate_dir(split),
        CFG.dssm_source_name: resolve_source_dir(CFG.dssm_source_name, split),
        CFG.bert_source_name: resolve_source_dir(CFG.bert_source_name, split),
    }
    if CFG.use_sasrec:
        source_map[CFG.sasrec_source_name] = resolve_source_dir(CFG.sasrec_source_name, split)
    for name, path in source_map.items():
        rows.append({
            "split": split,
            "source": name,
            "path": rel(path) if path else None,
            "exists": bool(path),
            "parquet_files": len(candidate_files(path)) if path else 0,
        })
    return pd.DataFrame(rows)

state = pd.concat([source_state(s) for s in ["ranker_train", "val", "test"]], ignore_index=True)
display(state)
state.to_parquet(REPORTS_DIR / f"{CFG.report_prefix}_source_state.parquet", index=False)
state.to_csv(REPORTS_DIR / f"{CFG.report_prefix}_source_state.csv", index=False)

,split,source,path,exists,parquet_files
0,ranker_train,base_non_dl,data\datamart\candidates\merged\ranker_train,True,128
1,ranker_train,dl_dssm_final,data\datamart\candidates\dl_dssm_final\ranker_...,True,5888
2,ranker_train,dl_bert4rec_small,data\datamart\candidates\dl_bert4rec_small\ran...,True,2688
3,ranker_train,dl_sasrec_v3,data\datamart\candidates\dl_sasrec_v3\ranker_t...,True,2688
4,val,base_non_dl,data\datamart\candidates\merged\val,True,128
5,val,dl_dssm_final,data\datamart\candidates\dl_dssm_final\val,True,13696
6,val,dl_bert4rec_small,data\datamart\candidates\dl_bert4rec_small\val,True,2560
7,val,dl_sasrec_v3,data\datamart\candidates\dl_sasrec_v3\val,True,2560
8,test,base_non_dl,data\datamart\candidates\merged\test,True,128
9,test,dl_dssm_final,data\datamart\candidates\dl_dssm_final\test,True,5504


## Ground truth and metrics

In [4]:
def scan_parquet_dir(path: Path, *, hive_partitioning: bool = True) -> pl.LazyFrame:
    if not path.exists() or not path_has_parquet(path):
        raise FileNotFoundError(path)
    return pl.scan_parquet(str(path / "**" / "*.parquet"), hive_partitioning=hive_partitioning)


def read_orders_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    path = AGG_DIR / "orders" / level
    lf = scan_parquet_dir(path, hive_partitioning=True)
    return lf.filter((pl.col("created_date").cast(pl.Date) >= pl.lit(start)) & (pl.col("created_date").cast(pl.Date) <= pl.lit(end)))


def build_ground_truth(start: date, end: date) -> dict[int, list[int]]:
    df = collect_streaming(
        read_orders_lf("user_item", start, end)
        .filter(pl.col("cnt_orders").fill_null(0) > 0)
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64))
        .unique()
        .sort(["user_id", "item_id"])
    )
    result: dict[int, list[int]] = defaultdict(list)
    for u, i in zip(df["user_id"].to_numpy(), df["item_id"].to_numpy()):
        result[int(u)].append(int(i))
    return dict(result)


def build_positive_pairs_df(gt: dict[int, list[int]]) -> pl.DataFrame:
    rows = [(int(u), int(i)) for u, items in gt.items() for i in items]
    if not rows:
        return pl.DataFrame({"user_id": [], "item_id": []}, schema={"user_id": pl.Int64, "item_id": pl.Int64})
    return pl.DataFrame(rows, schema={"user_id": pl.Int64, "item_id": pl.Int64}, orient="row").unique()


def _unique_int_list(values) -> list[int]:
    if values is None:
        return []
    return list(dict.fromkeys(int(x) for x in values))


def hit_rate_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    return float(any(int(x) in gt for x in pred_items[:k])) if gt else np.nan


def recall_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    return float(len(gt & set(int(x) for x in pred_items[:k])) / len(gt)) if gt else np.nan


def ndcg_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            dcg += 1.0 / np.log2(rank + 1)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, min(len(gt), k) + 1))
    return float(dcg / idcg) if idcg else 0.0


def ap_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    hits = 0
    score = 0.0
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            hits += 1
            score += hits / rank
    return float(score / len(gt))


def rr_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            return float(1.0 / rank)
    return 0.0

GT_VAL = build_ground_truth(CFG.val_target_start, CFG.val_target_end)
GT_PAIRS_VAL = build_positive_pairs_df(GT_VAL)
print(f"val users with GT: {len(GT_VAL):,}")
print(f"val positive pairs: {GT_PAIRS_VAL.height:,}")

val users with GT: 359,928
val positive pairs: 1,576,713


## Minimal source loading for merge-ablation

In [5]:
def _score_expr(cols: set[str], source: str) -> pl.Expr:
    for c in [f"{source}_score", "score", "max_source_score", "candidate_score"]:
        if c in cols:
            return pl.col(c).cast(pl.Float32)
    return pl.lit(0.0, dtype=pl.Float32)


def _rank_expr(cols: set[str], source: str, *, is_base: bool) -> pl.Expr:
    names = ["candidate_rank", "rank"] if is_base else ["rank", "candidate_rank", f"{source}_rank"]
    for c in names:
        if c in cols:
            return pl.col(c).cast(pl.Int32)
    raise ValueError(f"No rank column found for source={source}; columns={sorted(cols)[:30]}")


def load_source_shard_minimal(source_dir: Path | None, shard_id: int, source: str, *, max_rank: int, priority: int, is_base: bool) -> pl.DataFrame:
    if source_dir is None or max_rank <= 0:
        return pl.DataFrame(schema={
            "user_id": pl.Int64,
            "item_id": pl.Int64,
            "source_rank": pl.Int32,
            "source_score": pl.Float32,
            "source_priority": pl.Int8,
            "source_name": pl.Utf8,
        })
    lf = scan_candidate_shard_or_none(source_dir, shard_id)
    if lf is None:
        return pl.DataFrame(schema={
            "user_id": pl.Int64,
            "item_id": pl.Int64,
            "source_rank": pl.Int32,
            "source_score": pl.Float32,
            "source_priority": pl.Int8,
            "source_name": pl.Utf8,
        })
    cols = set(lazy_columns(lf))
    rank = _rank_expr(cols, source, is_base=is_base).alias("source_rank")
    score = _score_expr(cols, source).alias("source_score")
    return collect_streaming(
        lf.select(
            pl.col("user_id").cast(pl.Int64),
            pl.col("item_id").cast(pl.Int64),
            rank,
            score,
            pl.lit(priority, dtype=pl.Int8).alias("source_priority"),
            pl.lit(source).alias("source_name"),
        )
        .filter(pl.col("source_rank") <= int(max_rank))
    )


def merge_minimal_sources(frames: list[pl.DataFrame], *, total_cap: int) -> pl.DataFrame:
    frames = [f for f in frames if f is not None and not f.is_empty()]
    if not frames:
        return pl.DataFrame(schema={"user_id": pl.Int64, "item_id": pl.Int64, "candidate_rank": pl.Int32})
    df = pl.concat(frames, how="vertical_relaxed")
    # Duplicate user-item pairs can be present in several sources. Keep all source flags via source_list,
    # and sort by the strongest priority/rank/score.
    out = (
        df.group_by(["user_id", "item_id"])
        .agg([
            pl.col("source_priority").min().alias("best_priority"),
            pl.col("source_rank").min().alias("best_rank"),
            pl.col("source_score").max().alias("best_score"),
            pl.col("source_name").unique().alias("source_list"),
        ])
        .sort(
            ["user_id", "best_priority", "best_rank", "best_score", "item_id"],
            descending=[False, False, False, True, False],
        )
        .with_columns(pl.arange(1, pl.len() + 1).over("user_id").cast(pl.Int32).alias("candidate_rank"))
        .filter(pl.col("candidate_rank") <= int(total_cap))
        .select(["user_id", "item_id", "candidate_rank", "best_priority", "best_rank", "best_score", "source_list"])
    )
    return out

## In-memory validation merge experiments

In [6]:
def evaluate_predictions_from_shards(experiment_rows: list[dict], gt: dict[int, list[int]], ks: Iterable[int]) -> pd.DataFrame:
    ks = tuple(int(k) for k in ks)
    accum: dict[str, dict[int, defaultdict[str, float]]] = {}
    users_seen: dict[str, int] = defaultdict(int)
    candidate_counts: dict[str, list[int]] = defaultdict(list)
    unique_items: dict[str, set[int]] = defaultdict(set)

    for row in experiment_rows:
        name = row["name"]
        accum[name] = {k: defaultdict(float) for k in ks}

    for shard_id in trange(CFG.n_shards, desc="val merge/eval shards"):
        source_dirs = {
            CFG.base_source_name: resolve_base_candidate_dir("val"),
            CFG.dssm_source_name: resolve_source_dir(CFG.dssm_source_name, "val"),
            CFG.bert_source_name: resolve_source_dir(CFG.bert_source_name, "val"),
        }
        if CFG.use_sasrec:
            source_dirs[CFG.sasrec_source_name] = resolve_source_dir(CFG.sasrec_source_name, "val")

        # Load max needed once per shard.
        max_base = max(r["base_k"] for r in experiment_rows)
        max_dssm = max(r["dssm_k"] for r in experiment_rows)
        max_bert = max(r["bert_k"] for r in experiment_rows)
        max_sasrec = max(r.get("sasrec_k", 0) for r in experiment_rows)

        base_df = load_source_shard_minimal(source_dirs[CFG.base_source_name], shard_id, CFG.base_source_name, max_rank=max_base, priority=0, is_base=True)
        if base_df.is_empty():
            continue
        base_users_lf = base_df.select(pl.col("user_id").unique()).lazy()
        dssm_df = load_source_shard_minimal(source_dirs[CFG.dssm_source_name], shard_id, CFG.dssm_source_name, max_rank=max_dssm, priority=1, is_base=False)
        bert_df = load_source_shard_minimal(source_dirs[CFG.bert_source_name], shard_id, CFG.bert_source_name, max_rank=max_bert, priority=2, is_base=False)
        if not dssm_df.is_empty():
            dssm_df = dssm_df.join(base_users_lf.collect(), on="user_id", how="inner")
        if not bert_df.is_empty():
            bert_df = bert_df.join(base_users_lf.collect(), on="user_id", how="inner")
        sasrec_df = None
        if CFG.use_sasrec:
            sasrec_df = load_source_shard_minimal(source_dirs[CFG.sasrec_source_name], shard_id, CFG.sasrec_source_name, max_rank=max_sasrec, priority=3, is_base=False)
            if not sasrec_df.is_empty():
                sasrec_df = sasrec_df.join(base_users_lf.collect(), on="user_id", how="inner")

        for row in experiment_rows:
            frames = []
            if row["base_k"] > 0:
                frames.append(base_df.filter(pl.col("source_rank") <= int(row["base_k"])))
            if row["dssm_k"] > 0 and dssm_df is not None and not dssm_df.is_empty():
                frames.append(dssm_df.filter(pl.col("source_rank") <= int(row["dssm_k"])))
            if row["bert_k"] > 0 and bert_df is not None and not bert_df.is_empty():
                frames.append(bert_df.filter(pl.col("source_rank") <= int(row["bert_k"])))
            if CFG.use_sasrec and row.get("sasrec_k", 0) > 0 and sasrec_df is not None and not sasrec_df.is_empty():
                frames.append(sasrec_df.filter(pl.col("source_rank") <= int(row["sasrec_k"])))

            merged = merge_minimal_sources(frames, total_cap=int(row["total_cap"]))
            if merged.is_empty():
                continue
            pdf = merged.select(["user_id", "item_id", "candidate_rank"]).sort(["user_id", "candidate_rank"]).to_pandas()
            name = row["name"]
            max_k = max(ks)
            for user_id, group in pdf.groupby("user_id", sort=False):
                user_id = int(user_id)
                user_gt = gt.get(user_id)
                if not user_gt:
                    continue
                pred = group["item_id"].astype(np.int64).tolist()
                users_seen[name] += 1
                candidate_counts[name].append(len(pred))
                unique_items[name].update(pred[:max_k])
                for k in ks:
                    accum[name][k]["hit_rate"] += hit_rate_at_k(user_gt, pred, k)
                    accum[name][k]["recall"] += recall_at_k(user_gt, pred, k)
                    accum[name][k]["ndcg"] += ndcg_at_k(user_gt, pred, k)
                    accum[name][k]["map"] += ap_at_k(user_gt, pred, k)
                    accum[name][k]["mrr"] += rr_at_k(user_gt, pred, k)
            del merged, pdf
            gc.collect()
        del base_df, dssm_df, bert_df, sasrec_df
        gc.collect()

    rows = []
    for exp in experiment_rows:
        name = exp["name"]
        denom = max(users_seen[name], 1)
        for k in ks:
            rows.append({
                "model": name,
                "k": k,
                "users_eval": int(users_seen[name]),
                "hit_rate": accum[name][k]["hit_rate"] / denom,
                "recall": accum[name][k]["recall"] / denom,
                "ndcg": accum[name][k]["ndcg"] / denom,
                "map": accum[name][k]["map"] / denom,
                "mrr": accum[name][k]["mrr"] / denom,
                "avg_candidates_per_user": float(np.mean(candidate_counts[name])) if candidate_counts[name] else 0.0,
                "p50_candidates_per_user": float(np.percentile(candidate_counts[name], 50)) if candidate_counts[name] else 0.0,
                "p95_candidates_per_user": float(np.percentile(candidate_counts[name], 95)) if candidate_counts[name] else 0.0,
                "unique_items": int(len(unique_items[name])),
                "base_k": int(exp["base_k"]),
                "dssm_k": int(exp["dssm_k"]),
                "bert_k": int(exp["bert_k"]),
                "sasrec_k": int(exp.get("sasrec_k", 0)),
                "total_cap": int(exp["total_cap"]),
            })
    return pd.DataFrame(rows)


experiment_rows = []
for name, base_k, dssm_k, bert_k, sasrec_k, total_cap in CFG.experiment_grid:
    if sasrec_k and not CFG.use_sasrec:
        continue
    if total_cap is None:
        total_cap = base_k + dssm_k + bert_k + sasrec_k
    experiment_rows.append({
        "name": name,
        "base_k": int(base_k),
        "dssm_k": int(dssm_k),
        "bert_k": int(bert_k),
        "sasrec_k": int(sasrec_k),
        "total_cap": int(total_cap),
    })

pd.DataFrame(experiment_rows)

,name,base_k,dssm_k,bert_k,sasrec_k,total_cap
0,base_700,700,0,0,0,700
1,base700_dssm300,700,300,0,0,1000
2,base700_bert300,700,0,300,0,1000
3,base700_sasrec300,700,0,0,300,1000
4,base700_dssm300_bert50,700,300,50,0,1050
5,base700_dssm300_bert100,700,300,100,0,1100
6,base700_dssm300_bert200,700,300,200,0,1200
7,base700_dssm300_bert300,700,300,300,0,1300
8,base700_dssm300_sasrec50,700,300,0,50,1050
9,base700_dssm300_sasrec100,700,300,0,100,1100


In [7]:
val_metrics = evaluate_predictions_from_shards(experiment_rows, GT_VAL, ks=CFG.metric_ks)
display(val_metrics)
val_metrics.to_parquet(REPORTS_DIR / f"{CFG.report_prefix}_val_metrics.parquet", index=False)
val_metrics.to_csv(REPORTS_DIR / f"{CFG.report_prefix}_val_metrics.csv", index=False)
print("saved:", rel(REPORTS_DIR / f"{CFG.report_prefix}_val_metrics.parquet"))

val merge/eval shards:   0%|          | 0/128 [00:00<?, ?it/s]

,model,k,users_eval,hit_rate,recall,ndcg,map,mrr,avg_candidates_per_user,p50_candidates_per_user,p95_candidates_per_user,unique_items,base_k,dssm_k,bert_k,sasrec_k,total_cap
0,base_700,20,359928,0.030059,0.012853,0.006312,0.003013,0.006905,698.320442,700.0,700.0,3426893,700,0,0,0,700
1,base_700,50,359928,0.061157,0.023976,0.009198,0.003376,0.007839,698.320442,700.0,700.0,3426893,700,0,0,0,700
2,base_700,100,359928,0.110186,0.042449,0.013222,0.003664,0.008524,698.320442,700.0,700.0,3426893,700,0,0,0,700
3,base_700,200,359928,0.164541,0.064736,0.017515,0.003857,0.008917,698.320442,700.0,700.0,3426893,700,0,0,0,700
4,base_700,300,359928,0.195547,0.077662,0.019835,0.003926,0.009045,698.320442,700.0,700.0,3426893,700,0,0,0,700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,base700_dssm200_bert100_sasrec100,1000,359928,0.280259,0.118668,0.025577,0.003392,0.007930,1064.253012,1081.0,1098.0,3433374,700,200,100,100,1100
212,base700_dssm200_bert100_sasrec100,1100,359928,0.282073,0.119509,0.025701,0.003393,0.007932,1064.253012,1081.0,1098.0,3433374,700,200,100,100,1100
213,base700_dssm200_bert100_sasrec100,1200,359928,0.282073,0.119509,0.025701,0.003393,0.007932,1064.253012,1081.0,1098.0,3433374,700,200,100,100,1100
214,base700_dssm200_bert100_sasrec100,1300,359928,0.282073,0.119509,0.025701,0.003393,0.007932,1064.253012,1081.0,1098.0,3433374,700,200,100,100,1100


saved: artifacts\reports\dl_four_source_merge_val_metrics.parquet


## Unique positives / quota contribution

In [8]:
def collect_found_positive_pairs_at_k(path: Path | None, gt_pairs: pl.DataFrame, k: int, *, rank_col: str, source: str) -> set[tuple[int, int]]:
    found: set[tuple[int, int]] = set()
    if path is None or not candidate_dir_looks_complete(path) or gt_pairs.is_empty() or k <= 0:
        return found
    gt_lf = gt_pairs.lazy()
    for shard_id in trange(CFG.n_shards, desc=f"positive pairs {source}@{k}"):
        lf = scan_candidate_shard_or_none(path, shard_id)
        if lf is None:
            continue
        cols = set(lazy_columns(lf))
        if rank_col not in cols:
            rank_col2 = "rank" if "rank" in cols else "candidate_rank"
        else:
            rank_col2 = rank_col
        hits = collect_streaming(
            lf.select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), pl.col(rank_col2).cast(pl.Int32).alias("r"))
            .filter(pl.col("r") <= int(k))
            .join(gt_lf, on=["user_id", "item_id"], how="inner")
            .select("user_id", "item_id")
            .unique()
        )
        if not hits.is_empty():
            found.update((int(u), int(i)) for u, i in zip(hits["user_id"].to_numpy(), hits["item_id"].to_numpy()))
    return found

base_val_dir = resolve_base_candidate_dir("val")
dssm_val_dir = resolve_source_dir(CFG.dssm_source_name, "val")
bert_val_dir = resolve_source_dir(CFG.bert_source_name, "val")
sasrec_val_dir = resolve_source_dir(CFG.sasrec_source_name, "val") if CFG.use_sasrec else None

total_pos = GT_PAIRS_VAL.height
base_pos = collect_found_positive_pairs_at_k(base_val_dir, GT_PAIRS_VAL, CFG.base_keep_per_user, rank_col="candidate_rank", source="base")
dssm_pos_by_k = {k: collect_found_positive_pairs_at_k(dssm_val_dir, GT_PAIRS_VAL, k, rank_col="rank", source=f"dssm{k}") for k in sorted({0, 50, 100, 200, 300} - {0})}
bert_pos_by_k = {k: collect_found_positive_pairs_at_k(bert_val_dir, GT_PAIRS_VAL, k, rank_col="rank", source=f"bert{k}") for k in sorted({0, 50, 100, 200, 300} - {0})}
sasrec_pos_by_k = {}
if CFG.use_sasrec:
    sasrec_pos_by_k = {k: collect_found_positive_pairs_at_k(sasrec_val_dir, GT_PAIRS_VAL, k, rank_col="rank", source=f"sasrec{k}") for k in sorted({0, 50, 100, 200, 300} - {0})}

positive pairs base@700:   0%|          | 0/128 [00:00<?, ?it/s]

positive pairs dssm50@50:   0%|          | 0/128 [00:00<?, ?it/s]

positive pairs dssm100@100:   0%|          | 0/128 [00:00<?, ?it/s]

positive pairs dssm200@200:   0%|          | 0/128 [00:00<?, ?it/s]

positive pairs dssm300@300:   0%|          | 0/128 [00:00<?, ?it/s]

positive pairs bert50@50:   0%|          | 0/128 [00:00<?, ?it/s]

... (3 outputs left out)

positive pairs sasrec50@50:   0%|          | 0/128 [00:00<?, ?it/s]

positive pairs sasrec100@100:   0%|          | 0/128 [00:00<?, ?it/s]

positive pairs sasrec200@200:   0%|          | 0/128 [00:00<?, ?it/s]

positive pairs sasrec300@300:   0%|          | 0/128 [00:00<?, ?it/s]

In [9]:
unique_rows = []
for exp in experiment_rows:
    dssm_pos = dssm_pos_by_k.get(exp["dssm_k"], set()) if exp["dssm_k"] else set()
    bert_pos = bert_pos_by_k.get(exp["bert_k"], set()) if exp["bert_k"] else set()
    sasrec_pos = sasrec_pos_by_k.get(exp["sasrec_k"], set()) if CFG.use_sasrec and exp.get("sasrec_k", 0) else set()
    union_all = base_pos | dssm_pos | bert_pos | sasrec_pos
    unique_rows.append({
        "model": exp["name"],
        "base_k": exp["base_k"],
        "dssm_k": exp["dssm_k"],
        "bert_k": exp["bert_k"],
        "sasrec_k": exp.get("sasrec_k", 0),
        "total_cap": exp["total_cap"],
        "target_positive_pairs": int(total_pos),
        "base_found_pairs": int(len(base_pos)),
        "dssm_found_pairs": int(len(dssm_pos)),
        "bert_found_pairs": int(len(bert_pos)),
        "sasrec_found_pairs": int(len(sasrec_pos)),
        "dssm_unique_vs_base": int(len(dssm_pos - base_pos)),
        "bert_unique_vs_base": int(len(bert_pos - base_pos)),
        "bert_unique_vs_base_dssm": int(len(bert_pos - base_pos - dssm_pos)),
        "dssm_unique_vs_base_bert": int(len(dssm_pos - base_pos - bert_pos)),
        "dl_overlap_dssm_bert": int(len(dssm_pos & bert_pos)),
        "union_found_pairs": int(len(union_all)),
        "incremental_pairs_vs_base": int(len(union_all - base_pos)),
        "base_pair_recall": float(len(base_pos) / total_pos) if total_pos else np.nan,
        "union_pair_recall": float(len(union_all) / total_pos) if total_pos else np.nan,
        "incremental_pair_recall_vs_base": float(len(union_all - base_pos) / total_pos) if total_pos else np.nan,
    })
unique_report = pd.DataFrame(unique_rows)
display(unique_report)
unique_report.to_parquet(REPORTS_DIR / f"{CFG.report_prefix}_unique_positives.parquet", index=False)
unique_report.to_csv(REPORTS_DIR / f"{CFG.report_prefix}_unique_positives.csv", index=False)
print("saved:", rel(REPORTS_DIR / f"{CFG.report_prefix}_unique_positives.parquet"))

,model,base_k,dssm_k,bert_k,sasrec_k,total_cap,target_positive_pairs,base_found_pairs,dssm_found_pairs,bert_found_pairs,...,dssm_unique_vs_base,bert_unique_vs_base,bert_unique_vs_base_dssm,dssm_unique_vs_base_bert,dl_overlap_dssm_bert,union_found_pairs,incremental_pairs_vs_base,base_pair_recall,union_pair_recall,incremental_pair_recall_vs_base
0,base_700,700,0,0,0,700,1576713,140798,0,0,...,0,0,0,0,0,140798,0,0.089298,0.089298,0.000000
1,base700_dssm300,700,300,0,0,1000,1576713,140798,18303,0,...,11457,0,0,11457,0,152255,11457,0.089298,0.096565,0.007266
2,base700_bert300,700,0,300,0,1000,1576713,140798,0,11933,...,0,6865,6865,0,0,147663,6865,0.089298,0.093652,0.004354
3,base700_sasrec300,700,0,0,300,1000,1576713,140798,0,0,...,0,0,0,0,0,145734,4936,0.089298,0.092429,0.003131
4,base700_dssm300_bert50,700,300,50,0,1050,1576713,140798,18303,4951,...,11457,1952,1552,11057,1228,153807,13009,0.089298,0.097549,0.008251
5,base700_dssm300_bert100,700,300,100,0,1100,1576713,140798,18303,7088,...,11457,3331,2689,10815,1717,154944,14146,0.089298,0.098270,0.008972
6,base700_dssm300_bert200,700,300,200,0,1200,1576713,140798,18303,9961,...,11457,5401,4397,10453,2321,156652,15854,0.089298,0.099354,0.010055
7,base700_dssm300_bert300,700,300,300,0,1300,1576713,140798,18303,11933,...,11457,6865,5636,10228,2688,157891,17093,0.089298,0.100139,0.010841
8,base700_dssm300_sasrec50,700,300,0,50,1050,1576713,140798,18303,0,...,11457,0,0,11457,0,153409,12611,0.089298,0.097297,0.007998
9,base700_dssm300_sasrec100,700,300,0,100,1100,1576713,140798,18303,0,...,11457,0,0,11457,0,154226,13428,0.089298,0.097815,0.008516


saved: artifacts\reports\dl_four_source_merge_unique_positives.parquet


## Optional: materialize selected final merge for reranker

По умолчанию выключено. Включать после просмотра `val_metrics` / `unique_report`.

Рекомендуемый первый вариант для проверки реранкером:

```python
CFG.materialize_selected_config = True
CFG.materialized_output_name = "merged_with_dl_all"
CFG.selected_base_k = 700
CFG.selected_dssm_k = 300
CFG.selected_bert_k = 200  # или 300, если размер pool допустим
CFG.selected_total_cap = 1200
```

In [7]:
def normalize_base_lf(lf: pl.LazyFrame) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))
    if "candidate_rank" not in cols:
        if "rank" in cols:
            lf = lf.with_columns(pl.col("rank").cast(pl.Int32).alias("candidate_rank"))
        else:
            raise ValueError("Base candidate shard has neither candidate_rank nor rank")
    return lf


def normalize_dl_lf(lf: pl.LazyFrame, source: str) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))
    exprs = []
    if "rank" not in cols and "candidate_rank" in cols:
        exprs.append(pl.col("candidate_rank").cast(pl.Int32).alias("rank"))
    if "score" not in cols and f"{source}_score" in cols:
        exprs.append(pl.col(f"{source}_score").cast(pl.Float32).alias("score"))
    if "source" not in cols:
        exprs.append(pl.lit(source).alias("source"))
    if f"{source}_score" not in cols and "score" in cols:
        exprs.append(pl.col("score").cast(pl.Float32).alias(f"{source}_score"))
    if f"{source}_rank" not in cols and "rank" in cols:
        exprs.append(pl.col("rank").cast(pl.Int32).alias(f"{source}_rank"))
    if f"in_{source}" not in cols:
        exprs.append(pl.lit(1, dtype=pl.Int8).alias(f"in_{source}"))
    if f"f_num__candidate__in_{source}" not in cols:
        exprs.append(pl.lit(1.0, dtype=pl.Float32).alias(f"f_num__candidate__in_{source}"))
    if f"f_num__candidate__{source}_score" not in cols:
        score_expr = pl.col(f"{source}_score") if f"{source}_score" in cols else pl.col("score")
        exprs.append(score_expr.cast(pl.Float32).alias(f"f_num__candidate__{source}_score"))
    if f"f_num__candidate__{source}_rank" not in cols:
        rank_expr = pl.col(f"{source}_rank") if f"{source}_rank" in cols else pl.col("rank")
        exprs.append(rank_expr.cast(pl.Float32).alias(f"f_num__candidate__{source}_rank"))
    if f"f_num__candidate__{source}_rr" not in cols:
        rank_expr = pl.col(f"{source}_rank") if f"{source}_rank" in cols else pl.col("rank")
        exprs.append((1.0 / rank_expr.cast(pl.Float32).clip(lower_bound=1.0)).cast(pl.Float32).alias(f"f_num__candidate__{source}_rr"))
    return lf.with_columns(exprs) if exprs else lf


def aggregate_full_frame(df: pl.DataFrame) -> pl.DataFrame:
    agg_exprs = []
    for c in df.columns:
        if c in {"user_id", "item_id"}:
            continue
        if c.startswith("in_") or c.startswith("f_num__candidate__in_") or c.endswith("_present"):
            agg_exprs.append(pl.col(c).fill_null(0).max().alias(c))
        elif c in {"candidate_rank", "rank", "min_source_rank"} or c.endswith("_rank"):
            agg_exprs.append(pl.col(c).min().alias(c))
        elif c in {"score", "max_source_score"} or c.endswith("_score") or c.endswith("_rr") or c.endswith("_prior") or c.endswith("_pref"):
            agg_exprs.append(pl.col(c).max().alias(c))
        elif c == "source":
            agg_exprs.append(pl.col(c).drop_nulls().first().alias(c))
        else:
            agg_exprs.append(pl.col(c).drop_nulls().first().alias(c))
    out = df.group_by(["user_id", "item_id"]).agg(agg_exprs)
    flag_cols = [c for c in out.columns if c.startswith("in_")]
    score_cols = [c for c in out.columns if c.endswith("_score") and not c.startswith("f_num__")]
    rank_cols = [c for c in out.columns if c.endswith("_rank") and not c.startswith("f_num__")]
    exprs = []
    if flag_cols:
        exprs.append(pl.sum_horizontal(*[pl.col(c).fill_null(0).cast(pl.Int16) for c in flag_cols]).cast(pl.Int16).alias("sources_count"))
    if score_cols:
        exprs.append(pl.max_horizontal(*[pl.col(c).fill_null(-1e9).cast(pl.Float32) for c in score_cols]).cast(pl.Float32).alias("max_source_score"))
    if rank_cols:
        exprs.append(pl.min_horizontal(*[pl.col(c).fill_null(1_000_000_000).cast(pl.Int32) for c in rank_cols]).cast(pl.Int32).alias("min_source_rank"))
    return out.with_columns(exprs) if exprs else out


def read_full_source_frame(source_dir: Path | None, shard_id: int, source: str, keep_k: int, priority: int, *, is_base: bool, base_users: pl.DataFrame | None = None) -> pl.DataFrame:
    if source_dir is None or keep_k <= 0:
        return pl.DataFrame()
    lf = scan_candidate_shard_or_none(source_dir, shard_id)
    if lf is None:
        return pl.DataFrame()
    if is_base:
        lf = normalize_base_lf(lf)
        rank_col = "candidate_rank"
    else:
        lf = normalize_dl_lf(lf, source)
        cols = set(lazy_columns(lf))
        rank_col = "rank" if "rank" in cols else "candidate_rank"
        if base_users is not None:
            lf = lf.join(base_users.lazy(), on="user_id", how="inner")
    df = collect_streaming(
        lf.filter(pl.col(rank_col).cast(pl.Int32) <= int(keep_k))
        .with_columns([
            pl.lit(priority, dtype=pl.Int8).alias("_source_priority"),
            pl.col(rank_col).cast(pl.Int32).alias("_source_rank_for_sort"),
        ])
    )
    return df


def materialize_selected_merge_split(split: str) -> dict:
    base_dir = resolve_base_candidate_dir(split)
    dssm_dir = resolve_source_dir(CFG.dssm_source_name, split)
    bert_dir = resolve_source_dir(CFG.bert_source_name, split)
    sasrec_dir = resolve_source_dir(CFG.sasrec_source_name, split) if CFG.use_sasrec else None
    if base_dir is None:
        return {"split": split, "status": "no_base", "rows": 0, "users": 0}
    out_dir = CANDIDATES_DIR / CFG.materialized_output_name / split
    if out_dir.exists():
        shutil.rmtree(out_dir)
    ensure_dir(out_dir)
    rows_total = 0
    users_total = set()
    for shard_id in trange(CFG.n_shards, desc=f"materialize {split}"):
        base = read_full_source_frame(base_dir, shard_id, CFG.base_source_name, CFG.selected_base_k, 0, is_base=True)
        if base.is_empty():
            continue
        base_users = base.select(pl.col("user_id").cast(pl.Int64).unique())
        frames = [base]
        dssm = read_full_source_frame(dssm_dir, shard_id, CFG.dssm_source_name, CFG.selected_dssm_k, 1, is_base=False, base_users=base_users)
        bert = read_full_source_frame(bert_dir, shard_id, CFG.bert_source_name, CFG.selected_bert_k, 2, is_base=False, base_users=base_users)
        if not dssm.is_empty():
            frames.append(dssm)
        if not bert.is_empty():
            frames.append(bert)
        if CFG.use_sasrec and CFG.selected_sasrec_k > 0:
            sasrec = read_full_source_frame(sasrec_dir, shard_id, CFG.sasrec_source_name, CFG.selected_sasrec_k, 3, is_base=False, base_users=base_users)
            if not sasrec.is_empty():
                frames.append(sasrec)

        all_cols = sorted(set().union(*(set(f.columns) for f in frames)))
        aligned = []
        for f in frames:
            missing = [c for c in all_cols if c not in f.columns]
            if missing:
                f = f.with_columns([pl.lit(None).alias(c) for c in missing])
            aligned.append(f.select(all_cols))
        merged = aggregate_full_frame(pl.concat(aligned, how="vertical_relaxed"))
        score_for_sort = "max_source_score" if "max_source_score" in merged.columns else "score"
        if score_for_sort not in merged.columns:
            merged = merged.with_columns(pl.lit(0.0, dtype=pl.Float32).alias("score"))
            score_for_sort = "score"
        # Sort base first, then DSSM, then BERT, then optional SASRec.
        # If duplicate item appears in base, aggregation will keep min _source_priority = 0.
        merged = (
            merged
            .with_columns([
                pl.col("_source_priority").fill_null(9).cast(pl.Int8),
                pl.col("_source_rank_for_sort").fill_null(1_000_000).cast(pl.Int32),
                pl.col(score_for_sort).fill_null(-1e9).cast(pl.Float32).alias("_tmp_score"),
            ])
            .sort(
                ["user_id", "_source_priority", "_source_rank_for_sort", "_tmp_score", "item_id"],
                descending=[False, False, False, True, False],
            )
            .with_columns(pl.arange(1, pl.len() + 1).over("user_id").cast(pl.Int32).alias("candidate_rank"))
            .filter(pl.col("candidate_rank") <= int(CFG.selected_total_cap))
            .drop(["_tmp_score", "_source_priority", "_source_rank_for_sort"], strict=False)
        )
        shard_out = out_dir / f"user_shard={shard_id}"
        ensure_dir(shard_out)
        merged.write_parquet(shard_out / "part.parquet", compression="zstd")
        rows_total += merged.height
        users_total.update(int(u) for u in merged.select("user_id").unique().to_series().to_list())
        del frames, aligned, merged, base
        gc.collect()
    meta = {
        "split": split,
        "status": "merged",
        "out_dir": rel(out_dir),
        "rows": int(rows_total),
        "users": int(len(users_total)),
        "base_k": int(CFG.selected_base_k),
        "dssm_k": int(CFG.selected_dssm_k),
        "bert_k": int(CFG.selected_bert_k),
        "sasrec_k": int(CFG.selected_sasrec_k),
        "total_cap": int(CFG.selected_total_cap),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    (out_dir / "_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    return meta

In [8]:
if CFG.materialize_selected_config:
    materialize_rows = []
    for split in CFG.materialize_splits:
        materialize_rows.append(materialize_selected_merge_split(split))
    materialize_report = pd.DataFrame(materialize_rows)
    display(materialize_report)
    materialize_report.to_parquet(REPORTS_DIR / f"{CFG.report_prefix}_materialize_report.parquet", index=False)
    materialize_report.to_csv(REPORTS_DIR / f"{CFG.report_prefix}_materialize_report.csv", index=False)
else:
    print("Materialization disabled. Set CFG.materialize_selected_config=True after choosing quotas.")

materialize ranker_train:   0%|          | 0/128 [00:00<?, ?it/s]

materialize val:   0%|          | 0/128 [00:00<?, ?it/s]

materialize test:   0%|          | 0/128 [00:00<?, ?it/s]

,split,status,out_dir,rows,users,base_k,dssm_k,bert_k,sasrec_k,total_cap,built_at_utc
0,ranker_train,merged,data\datamart\candidates\merged_with_dl_all\ra...,472050365,376016,700,300,200,100,1300,2026-05-11T06:45:44.244575+00:00
1,val,merged,data\datamart\candidates\merged_with_dl_all\val,450412159,359928,700,300,200,100,1300,2026-05-11T06:50:05.878405+00:00
2,test,merged,data\datamart\candidates\merged_with_dl_all\test,436438468,346759,700,300,200,100,1300,2026-05-11T06:53:56.199474+00:00


## Manifest

In [9]:
manifest = {
    "name": CFG.report_prefix,
    "built_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "config": {k: str(v) if isinstance(v, date) else v for k, v in asdict(CFG).items()},
    "reports": {
        "source_state": rel(REPORTS_DIR / f"{CFG.report_prefix}_source_state.parquet"),
        "val_metrics": rel(REPORTS_DIR / f"{CFG.report_prefix}_val_metrics.parquet"),
        "unique_positives": rel(REPORTS_DIR / f"{CFG.report_prefix}_unique_positives.parquet"),
    },
}
if CFG.materialize_selected_config:
    manifest["materialized_output"] = {
        split: rel(CANDIDATES_DIR / CFG.materialized_output_name / split) for split in CFG.materialize_splits
    }
manifest_path = CANDIDATES_DIR / f"manifest_{CFG.report_prefix}.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", rel(manifest_path))

saved: data\datamart\candidates\manifest_dl_four_source_merge.json
